# Waze: predecir qué usuario abandona la aplicación

**Curso 4 del certificado, proyecto de regresión logística binomial.**

Waze quiere crecer, y para eso necesita que la gente no se vaya. El encargo es un modelo
que señale a quién está a punto de abandonar, para poder actuar antes.

**Lo que espero, escrito antes de ajustar nada para no poder ajustarlo después:** solo el
17,7 % de los usuarios abandona, las variables describen comportamiento y no causas, y en
el archivo no hay ni un dato sobre por qué se fue nadie. Lo probable es que el modelo deje
escapar a la mayoría de los que se van. Si pasa, se reporta.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "projects" / "curso4").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "projects"))
sys.path.insert(0, str(ROOT / "projects" / "curso4" / "waze" / "02_scripts"))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from waze_logistic import add_features, load_and_check_missing, vif_table
from common import Results, dataset

pd.set_option("display.width", 120)
print("listo")

listo


## 1. Las 700 etiquetas que faltan, y por qué se pueden tirar

Excluir a un grupo solo es inocuo si ese grupo se parece al resto. Se comprueba con
medianas, que no se mueven con los valores extremos de los que este archivo está lleno.

In [2]:
results = Results("waze", "Waze, cuaderno")
raw = pd.read_csv(dataset("waze"))
df = add_features(load_and_check_missing(results), results)
df[["label", "sessions", "drives", "activity_days", "driving_days", "device"]].head()


1. Los datos, y las 700 etiquetas que faltan
  filas en el CSV: 14999
  filas duplicadas: 0
  usuarios sin etiqueta: 700
    en porcentaje: 4.7
  mayor diferencia de medianas entre los dos grupos (%): 6.2
  % de iPhone entre los que no tienen etiqueta: 63.9
  % de iPhone entre el resto: 64.5
  usuarios con etiqueta: 14299
  usuarios que abandonan: 2536
  tasa de abandono (%): 17.74

2. Variables construidas, y la división por cero que esconden
  usuarios que condujeron cero días: 983
  kilómetros por día conducido, mediana: 273.3
  usuarios marcados como conductores profesionales: 2488
    su tasa de abandono (%): 7.56
    tasa del resto (%): 19.88


      label  sessions  drives  activity_days  driving_days   device
0  retained       283     226             28            19  Android
1  retained       133     107             13            11   iPhone
2  retained       114      95             14             8  Android
3  retained        49      40              7             3   iPhone
4  retained        84      68             27            18  Android


![Los usuarios sin etiqueta](03_figures/01_ausentes.png)

## 2. La división por cero que había escondida

983 usuarios condujeron cero días. Los kilómetros por día conducido dividen por ese número,
así que sin tratarlo el modelo recibe infinitos y falla sin decir por qué.

In [3]:
print(f"usuarios con cero dias conduciendo: {int(raw.driving_days.eq(0).sum())}")
print(f"infinitos en km por dia: {int(np.isinf(df.km_per_driving_day).sum())}")
print()
print("tasa de abandono por tipo de usuario")
print((df.groupby("professional_driver").churned.mean() * 100).round(2).to_string())

usuarios con cero dias conduciendo: 1024
infinitos en km por dia: 0

tasa de abandono por tipo de usuario
professional_driver
0    19.88
1     7.56


Los conductores profesionales abandonan al 7,56 % frente al 19,88 % del resto. Parece el
mejor predictor del archivo. Guárdalo, porque más abajo desaparece.

## 3. Dónde está la señal

In [4]:
print("dias de actividad en el ultimo mes, mediana")
print(df.groupby("label").activity_days.median().to_string())

dias de actividad en el ultimo mes, mediana
label
churned      8.0
retained    17.0


![Actividad por grupo](03_figures/02_actividad.png)

## 4. Multicolinealidad, y aquí sí es grave

Sesiones y trayectos correlacionan a 0,997. Días activo y días conduciendo, a 0,948. Son la
misma información contada dos veces, y el modelo no puede repartir el efecto entre ellas.

In [5]:
everything = ["sessions", "drives", "total_sessions", "n_days_after_onboarding",
              "driven_km_drives", "duration_minutes_drives", "activity_days",
              "driving_days", "km_per_driving_day", "professional_driver", "iphone"]
print(vif_table(df[everything]).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

               variable    vif
               sessions 159.30
                 drives 159.07
           driving_days  10.22
          activity_days   9.85
       driven_km_drives   2.13
duration_minutes_drives   1.95
    professional_driver   1.61
         total_sessions   1.54
     km_per_driving_day   1.42
                 iphone   1.00
n_days_after_onboarding   1.00


![Multicolinealidad](03_figures/03_multicolinealidad.png)

## 5. El modelo, con los coeficientes en unidades que se puedan imaginar

Por unidad suelta casi todas las razones de momios salen 1,000 aunque su valor p sea de
3e-48. El problema es la unidad, no el efecto: un día entre dos mil desde el alta, un
minuto dentro de una hora al volante.

In [6]:
features = ["drives", "total_sessions", "n_days_after_onboarding",
            "duration_minutes_drives", "activity_days", "km_per_driving_day",
            "professional_driver", "iphone"]
train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df.churned)
model = sm.Logit(train.churned, sm.add_constant(train[features])).fit(disp=False)

units = {"drives": ("cada 10 trayectos", 10), "total_sessions": ("cada 10 sesiones", 10),
         "n_days_after_onboarding": ("cada anio de antiguedad", 365),
         "duration_minutes_drives": ("cada hora al volante", 60),
         "activity_days": ("cada dia de actividad", 1),
         "km_per_driving_day": ("cada 100 km por dia", 100),
         "professional_driver": ("conductor profesional", 1), "iphone": ("usa iPhone", 1)}

table = pd.DataFrame([
    {"variable": units[n][0],
     "razon de momios": round(float(np.exp(model.params[n] * units[n][1])), 4),
     "p": f"{model.pvalues[n]:.3g}"}
    for n in features
])
print(table.to_string(index=False))
print(f"\npseudo R2 = {model.prsquared:.4f}")

               variable  razon de momios         p
      cada 10 trayectos           1.0163   0.00132
       cada 10 sesiones           1.0028     0.247
cada anio de antiguedad           0.8638  3.09e-48
   cada hora al volante           1.0045  9.46e-05
  cada dia de actividad           0.9006 5.03e-139
    cada 100 km por dia           1.0015     0.497
  conductor profesional           0.9933     0.952
             usa iPhone           1.0166     0.773

pseudo R2 = 0.1348


**Dos lecturas que valen el proyecto entero.**

El conductor profesional, que en crudo abandonaba menos de la mitad que el resto, dentro del
modelo tiene una razón de momios de 0,993 con un p de 0,952: nada. Los días de actividad ya
recogían ese efecto. Era una variable de confusión disfrazada de hallazgo, que es justo lo
que enseña el módulo 3.

Y el dispositivo tampoco importa, con un p de 0,773. Es la misma conclusión del proyecto de
Waze del Curso 3, ahora controlando por otras siete variables.

![Razones de momios](03_figures/04_momios.png)

## 6. El resultado incómodo

In [7]:
probability = model.predict(sm.add_constant(test[features]))
predicted = (probability >= 0.5).astype(int)

print(classification_report(test.churned, predicted,
                            target_names=["se queda", "abandona"], digits=3))
missed = int(((predicted == 0) & (test.churned == 1)).sum())
print(f"se le escapan {missed} de los {int(test.churned.sum())} que abandonan")
print(f"contestar siempre 'se queda' acertaria el {100 * (1 - test.churned.mean()):.2f} %")

              precision    recall  f1-score   support

    se queda      0.832     0.983     0.901      2941
    abandona      0.505     0.082     0.141       634

    accuracy                          0.823      3575
   macro avg      0.669     0.532     0.521      3575
weighted avg      0.774     0.823     0.766      3575

se le escapan 582 de los 634 que abandonan
contestar siempre 'se queda' acertaria el 82.27 %


Acierta el 82,3 % y detecta al 8,2 % de los que se van. Ese 82,3 % es casi exactamente lo
que se conseguiría no haciendo nada.

## 7. Y sin embargo el modelo sí sabe algo

In [8]:
print(f"AUC = {roc_auc_score(test.churned, probability):.4f}")
print()
for cut in [0.50, 0.30, 0.20, 0.15]:
    marked = (probability >= cut).astype(int)
    caught = int(((marked == 1) & (test.churned == 1)).sum())
    hit_rate = caught / max(int(marked.sum()), 1)
    print(f"umbral {cut:.2f}: avisa de {int(marked.sum()):>4}, "
          f"detecta {caught:>3} de {int(test.churned.sum())}, "
          f"aciertan {100 * hit_rate:.1f} % de los avisos")

AUC = 0.7368

umbral 0.50: avisa de  103, detecta  52 de 634, aciertan 50.5 % de los avisos
umbral 0.30: avisa de  725, detecta 279 de 634, aciertan 38.5 % de los avisos
umbral 0.20: avisa de 1262, detecta 395 de 634, aciertan 31.3 % de los avisos
umbral 0.15: avisa de 1634, detecta 474 de 634, aciertan 29.0 % de los avisos


![Lo que se compra al mover el umbral](03_figures/05_umbral.png)

![Curva ROC](03_figures/06_roc.png)

## 8. Conclusión

El modelo **no sirve con el umbral por defecto**, y no porque sea inútil: su AUC de 0,737
dice que ordena a los usuarios por riesgo bastante mejor que el azar. Lo que está mal es
dónde se pone el corte.

Bajándolo a 0,20 detecta al **62,3 %** de los que se van, al precio de que dos de cada tres
avisos sean falsos. Si una campaña de retención cuesta poco por usuario, ese cambio sale a
cuenta, y es una decisión del negocio y no del modelo.

Lo que sí queda demostrado es dónde está el riesgo: **el usuario nuevo y poco activo**. Cada
año de antigüedad multiplica los momios de abandonar por 0,864 y cada día de actividad por
0,901.

El informe está en `04_reports/executive_summary.md` y los números en
`04_reports/model_results.json`.